# Bushfire Extent GDB — Clean and Insert into DB

This notebook builds a data pipeline for the National Historical Bushfire Extents GDB and writes cleaned data to the database.

## Cleaning rules
- read the `National_Historical_Bushfire_Extents_v4` layer from the GDB
- drop `SHAPE_Length` and `SHAPE_Area` (internal Esri fields, not needed)
- trim and normalize whitespace in all text fields
- convert blank strings to missing values
- parse `ignition_date` and `extinguish_date` as timezone-aware datetimes
- drop rows with missing `ignition_date` (undatable fires are not useful)
- drop rows where `area_ha <= 0` (zero or negative area is not meaningful)
- validate centroids are within Australia's bounding box; drop rows outside it
- round `area_ha` and `perim_km` to 2 decimal places
- extract centroid `lat` and `lon` from geometry, rounded to 6 decimal places
- dedup by `fire_id`, keeping the record with the largest `area_ha`
- remove remaining exact duplicate rows
- return a detailed cleaning summary reporting each drop reason and count


In [10]:
from pathlib import Path
import pandas as pd
import pyogrio

INPUT_GDB = Path("input/bushfire_extent.gdb")
LAYER = "National_Historical_Bushfire_Extents_v4"

INPUT_GDB

PosixPath('input/bushfire_extent.gdb')

In [11]:
# Preview layer info without loading data
info = pyogrio.read_info(INPUT_GDB, layer=LAYER)
print(f"Features : {info['features']:,}")
print(f"CRS      : {info['crs']}")
print(f"Fields   : {list(info['fields'])}")

Features : 345,345
CRS      : EPSG:4283
Fields   : ['fire_id', 'fire_name', 'ignition_date', 'capture_date', 'extinguish_date', 'fire_type', 'ignition_cause', 'capt_method', 'area_ha', 'perim_km', 'state', 'agency', 'SHAPE_Length', 'SHAPE_Area']


In [12]:
AUSTRALIA_BOUNDS = (113.2, -43.6, 153.6, -10.6)
#                   west   south   east   north

def clean_bushfire_extents(input_path: Path, layer: str):
    # Read GDB with geometry to extract centroids
    gdf = pyogrio.read_dataframe(
        input_path,
        layer=layer,
        columns=[
            "fire_id", "fire_name", "ignition_date", "capture_date",
            "extinguish_date", "fire_type", "ignition_cause", "capt_method",
            "area_ha", "perim_km", "state", "agency"
        ]
    )

    rows_before = len(gdf)
    dropped = {}

    # Extract centroid lat/lon from geometry
    gdf["centroid_lon"] = gdf.geometry.centroid.x.round(6)
    gdf["centroid_lat"] = gdf.geometry.centroid.y.round(6)

    # Drop geometry — work with flat dataframe from here
    df = gdf.drop(columns="geometry")

    # Clean all text columns: normalize whitespace and convert blanks to NaN
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        s = df[col].astype("string")
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
        s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df[col] = s

    # Parse datetime columns as timezone-aware
    for col in ["ignition_date", "extinguish_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

    # Round numeric fields
    for col in ["area_ha", "perim_km"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").round(2)

    # Drop rows with missing ignition_date
    mask = df["ignition_date"].isna()
    dropped["missing_ignition_date"] = int(mask.sum())
    df = df[~mask].copy()

    # Drop rows where area_ha is zero, negative, or missing
    mask = df["area_ha"].isna() | (df["area_ha"] <= 0)
    dropped["invalid_area_ha"] = int(mask.sum())
    df = df[~mask].copy()

    # Drop rows where centroid is outside Australia's bounding box
    lon_min, lat_min, lon_max, lat_max = AUSTRALIA_BOUNDS
    mask = (
        df["centroid_lon"].isna() | df["centroid_lat"].isna()
        | (df["centroid_lon"] < lon_min) | (df["centroid_lon"] > lon_max)
        | (df["centroid_lat"] < lat_min) | (df["centroid_lat"] > lat_max)
    )
    dropped["centroid_outside_australia"] = int(mask.sum())
    df = df[~mask].copy()

    # Remove exact duplicate rows
    before_exact = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    dropped["exact_duplicates"] = before_exact - len(df)

    summary = {
        "input_rows": rows_before,
        "output_rows": len(df),
        "total_dropped": rows_before - len(df),
        "dropped_by_reason": dropped,
        "column_count": df.shape[1],
    }
    return df, summary

In [13]:
clean_df, summary = clean_bushfire_extents(INPUT_GDB, LAYER)
summary

/var/folders/w4/l4h21lrj0d59d23sfzvj3t5m0000gn/T/ipykernel_44333/1521224411.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid_lon"] = gdf.geometry.centroid.x.round(6)
/var/folders/w4/l4h21lrj0d59d23sfzvj3t5m0000gn/T/ipykernel_44333/1521224411.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid_lat"] = gdf.geometry.centroid.y.round(6)
/var/folders/w4/l4h21lrj0d59d23sfzvj3t5m0000gn/T/ipykernel_44333/1521224411.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove 

{'input_rows': 345345,
 'output_rows': 267783,
 'total_dropped': 77562,
 'dropped_by_reason': {'missing_ignition_date': 331,
  'invalid_area_ha': 77038,
  'centroid_outside_australia': 28,
  'exact_duplicates': 165},
 'column_count': 14}

In [15]:
clean_df.head()


,fire_id,fire_name,ignition_date,capture_date,extinguish_date,fire_type,ignition_cause,capt_method,area_ha,perim_km,state,agency,centroid_lon,centroid_lat
0,<NA>,RECG ANU 1973,1920-01-01 00:00:00+00:00,<NA>,NaT,Unknown,Undetermined,<NA>,43205.0,113.0,ACT (Australian Capital Territory),ACT PSC,148.888895,-35.522131
1,<NA>,RECG ANU 1973,1926-01-01 00:00:00+00:00,<NA>,NaT,Unknown,Undetermined,<NA>,42551.0,139.0,ACT (Australian Capital Territory),ACT PSC,148.846916,-35.411622
2,<NA>,<NA>,1931-01-01 00:00:00+00:00,<NA>,NaT,Unknown,Undetermined,<NA>,705.0,13.0,ACT (Australian Capital Territory),ACT PSC,148.827200,-35.385858
3,<NA>,RECG ANU 1973,1933-01-01 00:00:00+00:00,<NA>,NaT,Unknown,Undetermined,<NA>,3610.0,33.0,ACT (Australian Capital Territory),ACT PSC,148.834945,-35.597974
4,<NA>,RECG ANU 1973,1933-01-01 00:00:00+00:00,<NA>,NaT,Unknown,Undetermined,<NA>,113.0,4.0,ACT (Australian Capital Territory),ACT PSC,148.793108,-35.540762


In [16]:
# Save intermediate CSV so GDB doesn't need to be re-read
OUTPUT_CSV = Path("input/bushfire_extents_clean.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(clean_df):,} rows to {OUTPUT_CSV}")

Saved 267,783 rows to input/bushfire_extents_clean.csv


In [17]:
# checks
print(f"Blank rows remaining : {int(clean_df.isna().all(axis=1).sum())}")
print(f"Exact duplicates remaining : {int(clean_df.duplicated().sum())}")
print(f"Missing ignition_date : {int(clean_df['ignition_date'].isna().sum())}")
print(f"Rows with area_ha <= 0 : {int((clean_df['area_ha'] <= 0).sum())}")
print(f"Missing centroid_lat : {int(clean_df['centroid_lat'].isna().sum())}")
print(f"Missing centroid_lon : {int(clean_df['centroid_lon'].isna().sum())}")
print(f"Whitespace-only text cells : {int(sum(clean_df[col].fillna('').str.fullmatch(r'\\s+').sum() for col in clean_df.select_dtypes(include='string').columns))}")
print()
print("State value counts:")
print(clean_df['state'].value_counts())
print()
print("Date range:")
print(f"ignition_date : {clean_df['ignition_date'].min()} → {clean_df['ignition_date'].max()}")

Blank rows remaining : 0
Exact duplicates remaining : 0
Missing ignition_date : 0
Rows with area_ha <= 0 : 0
Missing centroid_lat : 0
Missing centroid_lon : 0
Whitespace-only text cells : 0

State value counts:
state
WA (Western Australia)                150703
NSW (New South Wales)                  47549
VIC (Victoria)                         39360
QLD (Queensland)                       14778
TAS (Tasmania)                          7527
SA (South Australia)                    6165
ACT (Australian Capital Territory)      1701
Name: count, dtype: Int64

Date range:
ignition_date : 1898-01-01 00:00:00+00:00 → 2025-09-20 00:00:00+00:00


# Insert Clean Data into DB

In [20]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT")
DB_NAME     = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/bushfire_extents.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


In [21]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
df_insert = clean_df.copy()

target_columns = [
    "fire_id",
    "fire_name",
    "ignition_date",
    "area_ha",
    "perim_km",
    "state",
    "agency",
    "centroid_lat",
    "centroid_lon",
]

df_insert = df_insert[target_columns].copy()

# text columns
text_cols = ["fire_id", "fire_name", "state", "agency"]
for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

# float columns
float_cols = ["area_ha", "perim_km", "centroid_lat", "centroid_lon"]
for col in float_cols:
    df_insert[col] = pd.to_numeric(df_insert[col], errors="coerce")

In [22]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "bushfire_extents",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert):,} rows into bushfire_extents")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM bushfire_extents"))
    row_count = result.scalar()

print("Rows in table:", row_count)

Inserted 267,783 rows into bushfire_extents
Rows in table: 267783
